**First, please make sure to run the "catalog_setup" notebook and follow its instructions regarding uploading data into the volume**

In [0]:
USE CATALOG hospital_records_mavena

*OBJECTIVE 1: ENCOUNTERS OVERVIEW*

In [0]:
-- a. How many total encounters occurred each year?
SELECT YEAR(START) AS `Year`, COUNT(*) AS total_encounters FROM default.encounters
GROUP BY YEAR(START)
ORDER BY YEAR(START)

Year,total_encounters
2011,1336
2012,2106
2013,2495
2014,3885
2015,2469
2016,2451
2017,2360
2018,2292
2019,2228
2020,2519


In [0]:
-- b. For each year, what percentage of all encounters belonged to each encounter class
-- (ambulatory, outpatient, wellness, urgent care, emergency, and inpatient)?

SELECT YEAR(START) AS `Year`, COUNT(*) AS total_encounters,
  ROUND(SUM(CASE WHEN ENCOUNTERCLASS = 'ambulatory' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS ambulatory_percentage,
  ROUND(SUM(CASE WHEN ENCOUNTERCLASS = 'outpatient' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS outpatient_percentage,
  ROUND(SUM(CASE WHEN ENCOUNTERCLASS = 'wellness' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS wellness_percentage,
  ROUND(SUM(CASE WHEN ENCOUNTERCLASS = 'urgentcare' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS urgent_care_percentage,
  ROUND(SUM(CASE WHEN ENCOUNTERCLASS = 'emergency' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS emergency_percentage,
  ROUND(SUM(CASE WHEN ENCOUNTERCLASS = 'inpatient' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS inpatient_percentage 
  FROM default.encounters
GROUP BY YEAR(START) 
ORDER BY YEAR(START)


Year,total_encounters,ambulatory_percentage,outpatient_percentage,wellness_percentage,urgent_care_percentage,emergency_percentage,inpatient_percentage
2011,1336,49.93,24.48,13.02,2.25,4.12,6.21
2012,2106,42.50,21.08,9.12,14.20,8.69,4.42
2013,2495,44.33,19.44,7.45,14.39,9.02,5.37
2014,3885,60.26,17.86,4.86,8.42,5.56,3.04
2015,2469,43.46,20.49,6.93,15.39,9.23,4.50
2016,2451,43.78,19.62,7.43,13.91,10.20,5.06
2017,2360,41.82,20.13,7.16,16.31,9.24,5.34
2018,2292,40.71,20.94,7.59,16.45,10.78,3.53
2019,2228,37.97,20.47,7.54,17.82,10.19,6.01
2020,2519,47.32,19.73,6.31,14.49,9.29,2.86


In [0]:
-- c. What percentage of encounters were over 24 hours versus under 24 hours?

SELECT COUNT(*) AS total_encounters,
  ROUND(SUM(CASE WHEN DATEDIFF(HOUR, START, STOP) >= 24 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS over_24_percentage,
  ROUND(SUM(CASE WHEN DATEDIFF(HOUR, START, STOP) < 24 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS under_24_percentage
  FROM default.encounters;

total_encounters,over_24_percentage,under_24_percentage
27891,4.13,95.87


*OBJECTIVE 2: COST & COVERAGE INSIGHTS*

In [0]:
-- a. How many encounters had zero payer coverage, and what percentage of total encounters does this represent?

SELECT COUNT(*) AS total_encounters, 
  SUM(CASE WHEN payer_coverage = 0 THEN 1 ELSE 0 END) AS zero_payer_coverage_total,
  ROUND(SUM(CASE WHEN payer_coverage = 0 THEN 1 ELSE 0 END) / COUNT(*) * 100.0, 2) AS zero_payer_coverage_percentage
  FROM default.encounters

total_encounters,zero_payer_coverage_total,zero_payer_coverage_percentage
27891,13586,48.71


In [0]:
-- b. What are the top 10 most frequent procedures performed and the average base cost for each?
SELECT CODE, Description, COUNT(*) AS total_procedures, ROUND(AVG(Base_cost), 2) as average_base_cost FROM default.procedures
GROUP BY CODE, Description
ORDER BY COUNT(*) DESC
LIMIT 10

CODE,Description,total_procedures,average_base_cost
710824005,Assessment of health and social care needs (procedure),4596,431.0
385763009,Hospice care (regime/therapy),4098,431.0
171207006,Depression screening (procedure),3614,431.0
454711000124102,Depression screening using Patient Health Questionnaire Two-Item score (procedure),3614,431.0
428211000124100,Assessment of substance use (procedure),2906,431.0
265764009,Renal dialysis (procedure),2746,1004.09
762993000,Assessment using Morse Fall Scale (procedure),2422,431.0
710841007,Assessment of anxiety (procedure),2288,431.0
430193006,Medication Reconciliation (procedure),2284,509.12
713106006,Screening for drug abuse (procedure),1484,431.0


In [0]:
-- c. What are the top 10 procedures with the highest average base cost and the number of times they were performed?
SELECT CODE, Description, ROUND(AVG(Base_cost), 2) as average_base_cost, COUNT(*) AS total_procedures 
FROM default.procedures
GROUP BY CODE, Description
ORDER BY average_base_cost DESC
LIMIT 10

CODE,Description,average_base_cost,total_procedures
305351004,Admit to ICU (procedure),206260.4,5
232717009,Coronary artery bypass grafting,47085.89,9
392021009,Lumpectomy of breast (procedure),29353.0,5
302497006,Hemodialysis (procedure),29299.56,27
447365002,Insertion of biventricular implantable cardioverter defibrillator,27201.0,4
180325003,Electrical cardioversion,25903.11,1383
43075005,Partial resection of colon,25229.29,7
432231006,Fine needle aspiration biopsy of lung (procedure),23141.0,1
433112001,Percutaneous mechanical thrombectomy of portal vein using fluoroscopic guidance,20228.04,57
415070008,Percutaneous coronary intervention,19728.0,9


In [0]:
-- d. What is the average total claim cost for encounters, broken down by payer?
SELECT p.name, ROUND(AVG(total_claim_cost), 2) AS average_total_claim_cost FROM encounters e
LEFT JOIN payers p
ON e.Payer = p.id
GROUP BY p.name
ORDER BY average_total_claim_cost DESC

name,average_total_claim_cost
Medicaid,6205.22
NO_INSURANCE,5593.2
Anthem,4236.81
Humana,3269.3
Blue Cross Blue Shield,3245.58
Cigna Health,2996.95
UnitedHealthcare,2848.34
Aetna,2767.05
Medicare,2167.55
Dual Eligible,1696.19


*OBJECTIVE 3: PATIENT BEHAVIOR ANALYSIS*

In [0]:
-- a. How many unique patients were admitted each quarter over time?
SELECT CONCAT(YEAR(START),"Q",QUARTER(START)) as year_quarter, 
  COUNT(DISTINCT(patient)) AS num_distinct_patients
FROM default.encounters
GROUP BY year_quarter
ORDER BY year_quarter
LIMIT 10;

year_quarter,num_distinct_patients
2011Q1,156
2011Q2,162
2011Q3,155
2011Q4,168
2012Q1,249
2012Q2,261
2012Q3,232
2012Q4,240
2013Q1,243
2013Q2,271


In [0]:
-- b. How many patients were readmitted within 30 days of a previous encounter?
WITH readmissions AS (
  SELECT patient, START, STOP,
    LEAD(START) OVER(PARTITION BY patient ORDER BY START) AS next_start_date
    FROM encounters) 

SELECT COUNT(DISTINCT patient) AS num_patients
FROM readmissions
WHERE DATEDIFF(next_start_date, STOP) < 30;

count(DISTINCT patient)
771


In [0]:
-- c. Which patients had the most readmissions?
WITH readmissions AS (
  SELECT patient, START, STOP,
    LEAD(START) OVER(PARTITION BY patient ORDER BY START) AS next_start_date
    FROM encounters) 

SELECT patient, COUNT(*) as total_readmissions
FROM readmissions
WHERE DATEDIFF(next_start_date, STOP) < 30
GROUP BY patient
ORDER BY COUNT(*) DESC
LIMIT 5;

patient,total_readmissions
1712d26d-822d-1e3a-2267-0a9dba31d7c8,1376
3de74169-7f67-9304-91d4-757e0f3a14d2,876
5e055638-0dad-dfd5-005d-1e74b6fd29ac,871
3f523789-55f3-bb31-2757-4803ca6a9c2a,442
5dcb295d-92df-a147-ebcc-aa49b6262830,421
